Step 1: Bronze Auto Loader Pipeline Script.

Run this cell to ingest all generated JSON batches incrementally into workspace.default.bronze_clickstream_events:

In [0]:
# File: src/bronze/ingest_raw_events.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp

class BronzeClickstreamIngestion:
    def __init__(self, catalog: str = "workspace", schema: str = "default", volume: str = "raw_data"):
        self.spark = SparkSession.builder.getOrCreate()
        
        # Exact Storage Endpoints aligned with Data Generator
        self.landing_path = f"/Volumes/{catalog}/{schema}/{volume}/clickstream_landing/"
        self.bronze_table = f"{catalog}.{schema}.bronze_clickstream_events"
        self.checkpoint_path = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoints/clickstream_events/"
        self.schema_path = f"/Volumes/{catalog}/{schema}/{volume}/_schemas/clickstream_events/"

    def run_pipeline(self):
        print(f"Starting Auto Loader Stream from: {self.landing_path}")

        # 1. Read Stream via Auto Loader with Rescued Data Tracking
        df_raw = (
            self.spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation", self.schema_path)
            .option("cloudFiles.rescuedDataColumn", "_rescued_data")
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .load(self.landing_path)
        )

        # 2. Bronze Transformations (Metadata & Lineage)
        df_bronze = (
            df_raw
            .withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", col("_metadata.file_path"))
        )

        # 3. Write Incremental Stream to Delta Table
        query = (
            df_bronze.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", self.checkpoint_path)
            .option("mergeSchema", "true")
            .trigger(availableNow=True)
            .table(self.bronze_table)
        )
        
        query.awaitTermination()
        print(f"Ingestion finished successfully into table: {self.bronze_table}")

if __name__ == "__main__":
    pipeline = BronzeClickstreamIngestion()
    pipeline.run_pipeline()

generate new batches of data (designed to test and simulate incremental data processing)


In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import time

LANDING_PATH = "/Volumes/workspace/default/raw_data/clickstream_landing"

def generate_single_batch(records_per_batch=2000, inject_bad_records=True):
    timestamp_id = int(time.time())
    
    df = (
        spark.range(0, records_per_batch)
        .withColumn("event_id", F.expr("uuid()"))
        .withColumn("user_id", F.concat(F.lit("USR_"), (F.rand() * 1000 + 1).cast("int")))
        .withColumn("session_id", F.expr("uuid()"))
        .withColumn("product_id", F.concat(F.lit("PROD_"), (F.rand() * 50 + 1).cast("int")))
        .withColumn("event_type", F.element_at(F.array(F.lit("view"), F.lit("cart"), F.lit("purchase")), (F.rand() * 3 + 1).cast("int")))
        .withColumn("quantity", (F.rand() * 5 + 1).cast("int"))
        .withColumn("unit_price", F.round(F.rand() * 100 + 5, 2))
        .withColumn("event_timestamp", F.current_timestamp())
    )
    
    if inject_bad_records:
        current_time = datetime.now()
        dirty_df = spark.createDataFrame([
            (9999, "", "USR_999", "sess_bad", "PROD_1", "purchase", -5, 10.0, current_time)
        ], df.schema)
        df = df.union(dirty_df)

    output_file = f"{LANDING_PATH}/batch_{timestamp_id}.json"
    df.coalesce(1).write.mode("overwrite").json(output_file)

# Run to push a new batch to landing volume
generate_single_batch()

Here are the three easiest and most reliable ways to verify the incremental data load of an auto-loader.

1. Source File Grouping Query

Group by the _source_file column in your Bronze table to inspect which specific JSON files have been processed:

In [0]:
%sql
SELECT 
    _source_file, 
    COUNT(*) as total_records, 
    MIN(_ingested_at) as first_ingested
FROM workspace.default.bronze_clickstream_events
GROUP BY _source_file
ORDER BY first_ingested DESC;

Verification: When you run **generate_new_batch** followed by **ingest_raw_events**, only the new batch file will appear as a new row in the results. Existing files will not be re-ingested. 

2. Count Before & After Method

Follow these steps to practically test the pipeline's incremental loading:

Step A: Check the current total record count:
Step B: Run **ingest_raw_events** again without generating any new data.

Step C: Re-check the count. It should remain unchanged (the checkpoint ensures previously processed files are skipped).

Step D: Run generate_new_batch to generate new data, then execute **ingest_raw_events**. The count should increase.

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.bronze_clickstream_events;

3. Checkpoint Offsets Directory Inspection

Verify the Auto Loader offset history directly using DBUtils:

Verification: Each time the pipeline runs and processes new data, sequential numeric files (0, 1, 2, etc.) are written to the offsets directory. These files serve as the audit log of all processed files and offsets.

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw_data/_checkpoints/clickstream_events/offsets"))